# PoliMillionaire - BM25 retrieval, no LLM

Runs all competitions through the API using the local SimpleWiki BM25 index. Each competition is played 5 times and every question attempt is printed and saved to CSV.

In [1]:
from pathlib import Path
import os
import sys

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / "api_client" / "NLP_assignment_api_client").exists() and (path / "project" / "src").exists():
            return path
    raise FileNotFoundError("Could not find project root")

PROJECT_ROOT = find_project_root()
CLIENT_PARENT = PROJECT_ROOT / "api_client" / "NLP_assignment_api_client"
SRC_DIR = PROJECT_ROOT / "project" / "src"

for path in [str(CLIENT_PARENT), str(SRC_DIR)]:
    if path not in sys.path:
        sys.path.insert(0, path)

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\Tommaso\Code\NLP_polimi_26


In [2]:
from getpass import getpass
from millionaire_client import MillionaireClient
from retrieval_quiz_runner import load_retrieval_index, run_all_competitions, summarize, write_logs

API_URL = os.getenv("POLIMILLIONAIRE_API_URL", "http://131.175.15.22:51111/")
USERNAME = os.getenv("POLIMILLIONAIRE_USERNAME") or input("Username: ").strip()
PASSWORD = os.getenv("POLIMILLIONAIRE_PASSWORD") or getpass("Password: ")

client = MillionaireClient(API_URL, timeout=30)
user = client.login(USERNAME, PASSWORD)
print(f"Logged in as {user.username} ({user.role})")

Logged in as NeuroniNegroni (student)


In [3]:
INDEX_PATH = PROJECT_ROOT / "data" / "indexes" / "simplewiki_160w_bm25.joblib"
if not INDEX_PATH.exists():
    raise FileNotFoundError(f"Missing index: {INDEX_PATH}")

index = load_retrieval_index(INDEX_PATH)
print("Loaded index:", INDEX_PATH)
print("Kind:", index["kind"])
print("Chunks:", len(index["docs"]))

Loaded index: C:\Users\Tommaso\Code\NLP_polimi_26\data\indexes\simplewiki_160w_bm25.joblib
Kind: bm25
Chunks: 434093


In [4]:
competitions = client.competitions.list_all()
for comp in competitions:
    print(f"{comp.id}: {comp.name} | max levels: {comp.max_levels} | questions: {comp.question_count}")

0: Entertainment | max levels: 15 | questions: 843
1: Ancient History and Politics | max levels: 15 | questions: 456
2: Science and Nature | max levels: 15 | questions: 5395
3: Maths | max levels: 15 | questions: 390


In [ ]:
ATTEMPTS_PER_COMPETITION = 5
STRATEGY_NAME = "simplewiki_bm25_no_llm"

rows = run_all_competitions(
    client=client,
    index=index,
    strategy_name=STRATEGY_NAME,
    attempts_per_competition=ATTEMPTS_PER_COMPETITION,
)

output_path = PROJECT_ROOT / "logs" / "simplewiki_bm25_no_llm_all_competitions.csv"
write_logs(rows, output_path)
print(f"Wrote {len(rows)} rows to {output_path}")

[simplewiki_bm25_no_llm] comp=0 Entertainment attempt=1 q=1 level=0 chosen=3 correct=None earned=0 latency=32.10790190001717
Q: Which of the following best describes the impact of Orson Welles' first radio experience at the Todd Seminary for Boys?
A: It led to his immediate success as a radio star.
Top evidence: Orson Welles :: George Orson Welles (May 6, 1915 – October 10, 1985) was an American director, actor, writer, and producer who is remembered for his innovative work in film, radio, and theatre. He is considered to be among the greatest and most influential filmmakers of all time. Welles was born in Kenosha, Wisconsin. In 1938 Welles directed a weekly radio show...

[simplewiki_bm25_no_llm] comp=0 Entertainment attempt=2 q=1 level=0 chosen=2 correct=True earned=100 latency=22.267950699999346
Q: What is the primary purpose of the 'You Wouldn't Steal a Car' public service announcement trailers?
A: To raise awareness about copyright infringement
Top evidence: Copyright infringement

In [ ]:
summarize(rows)